In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

In [4]:
os.chdir(r'D:\AI\genai-learning-journey\02_deep_learning_for_NLP\ANN Projects\Customer Churn Classification')

In [5]:
df = pd.read_csv('data/processed_dataset.csv')

In [6]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [8]:
X = df.drop('Exited', axis = 1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
# Data Standardization

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]], shape=(8000, 12))

In [12]:
import pickle
with open('models/Standard_Scaler_model.pkl', 'wb') as file:
    pickle.dump(scaler, file)

### ANN Implementation

In [17]:
import tensorflow as tlf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
X_train.shape[1], # Number of input features to first layer

(12,)

In [24]:
model = Sequential([
    Dense(64, activation='relu', input_shape = (X_train.shape[1],)), # HL1 (Connected to OP)
    Dense(32, activation='relu'), # HL 2
    Dense(1, activation='sigmoid') # OP layer (1 output -> one neuron)
])

In [25]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
## Setup Optimizers and Loss Function
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()

In [27]:
# compiling model
model.compile(optimizer=opt, loss= loss, metrics=['accuracy'])

In [37]:
# Setting UP tensorBoard callback
from tensorflow.keras.callbacks import TensorBoard
from datetime import datetime
log_dir = 'logs/fit/' + datetime.now().strftime("%d_%m_%Y-%H_%M_%S")
TensorBoard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [35]:
# Setting Up EarlyStopping Callback
from tensorflow.keras.callbacks import EarlyStopping
EarlyStopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [38]:
# Train the Model
model.fit(
    X_train, y_train, validation_data = (X_test, y_test), epochs = 100,
    callbacks = [TensorBoard_callback, EarlyStopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8339 - loss: 0.3989 - val_accuracy: 0.8510 - val_loss: 0.3648
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8518 - loss: 0.3546 - val_accuracy: 0.8595 - val_loss: 0.3464
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8575 - loss: 0.3496 - val_accuracy: 0.8560 - val_loss: 0.3420
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8597 - loss: 0.3441 - val_accuracy: 0.8595 - val_loss: 0.3430
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8605 - loss: 0.3396 - val_accuracy: 0.8540 - val_loss: 0.3365
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8610 - loss: 0.3405 - val_accuracy: 0.8550 - val_loss: 0.3596
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8629 - loss: 0.3363 - val_accuracy: 0.8570 - val_loss: 0.3376
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8641 - loss: 0.3346 - val_acc

In [39]:
## Load Tensorboard Extension
%load_ext tensorboard

In [40]:
%tensorboard --logdir logs/fit

In [41]:
# saving ANN Model
model.save('models/ann_classification_model.h5')